# Intra-Annotator Agreement Analysis

I created this notebook to compute intra-annotator reliability for my report writeup. I do this by comparing my original gold annotations against a blind re-annotation of the same 150-sentence sample, which I drew after a two-week gap. I use Cohen's quadratic weighted kappa (QWK) as the primary metric per dimension here as it accounts for the ordinal structure of my 0-3 scale by penalising larger discrepancies more heavily than disagreements between adjacent categories. These results are reported in the methodology section of my paper, which I use to establish that the gold annotations are a reliable data source for model training.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from scipy.stats import spearmanr

In [ ]:
# so everything can be changed about at the top here, I set it up as a config dictionary
agreement_config= {
    'gold_annotations_filename': 'manifesto_corpus_annotation_batches.xlsx',
    'reannotation_filename': 'reannotation_sample_blind.xlsx',
    'score_cols': ['threat_score', 'urgency_score', 'justification_score'],
    'likert_labels': [0, 1, 2, 3],
    'high_discrepancy_threshold': 2, # sentences disagreeing by this many Likert points or more get specifically flagged
}

## Load and merge annotations
1st all the gold batches get loaded in an combined from their seperate sheets to produce the full df of gold labels, like I do in model training too. Then I load in the blind reannotated batch of 150 sentences with the same method.

In [ ]:
# 1st I load my original gold batches & concatenate
all_gold_sheets= pd.read_excel(agreement_config['gold_annotations_filename'], sheet_name=None)
original_gold_df_for_comparison= pd.concat([gold_batch for gold_sheet_name, gold_batch in all_gold_sheets.items() if gold_sheet_name.startswith('Batch_')], ignore_index=True) # that's how I labelled them when I exported this file

for scoring_col in agreement_config['score_cols']:
    original_gold_df_for_comparison[scoring_col]= pd.to_numeric(original_gold_df_for_comparison[scoring_col], errors='coerce')
print(f"great, original gold annotations loaded, giving {len(original_gold_df_for_comparison)} sentences (should be 1500)")

# now I load blind re-annotations the same way
reannotated_df= pd.read_excel(agreement_config['reannotation_filename'])
for scoring_col in agreement_config['score_cols']:
    reannotated_df[scoring_col]= pd.to_numeric(reannotated_df[scoring_col], errors='coerce')
reannotated_df= reannotated_df[reannotated_df[agreement_config['score_cols']].notna().all(axis=1)].copy()
print(f"good, re-annotations loaded in too, giving {len(reannotated_df)} valid sentences to compare against gold.")

# now i merge on sentence_text col to align
gold_for_merge= original_gold_df_for_comparison[['sentence_text'] + agreement_config['score_cols']].drop_duplicates(subset='sentence_text', keep='first')
merged_annotations_df= reannotated_df[['sentence_text'] + agreement_config['score_cols']].merge(gold_for_merge, on='sentence_text', suffixes=('_reannot', '_original'))
print(f"there's {len(merged_annotations_df)} sentences matched for the comparison.")

## QWK and agreement metrics
Here I compute QWK as the primary reliability metric alongside Spearman rank correlation, exact agreement & within-one agreement too as supplementary checks. The latter of those gives me a practical lower bound on ordinal consistency that's robust to the sparsity of my high-intensity labels (which is so because high levels of securitsing is rare in manifesto sentences).


In [ ]:
per_dimension_agreement_results= {}

for dim in agreement_config['score_cols']:
    original_ratings= merged_annotations_df[f'{dim}_original'].astype(int).values
    reannot_ratings= merged_annotations_df[f'{dim}_reannotated'].astype(int).values

    qwk= cohen_kappa_score(original_ratings, reannot_ratings, weights='quadratic', labels=agreement_config['likert_labels'])
    spearman_rho, spearman_p= spearmanr(original_ratings, reannot_ratings)
    exact_agreement_pct= (original_ratings == reannot_ratings).mean()
    within_one_pct= (np.abs(original_ratings - reannot_ratings) <= 1).mean()

    per_dimension_agreement_results[dim]= {'qwk': qwk, 'spearman_rho': spearman_rho, 'spearman_p': spearman_p,
        'exact_agreement_pct': exact_agreement_pct, 'within_one_pct': within_one_pct}

    dim_label= dim.replace('_score', '').capitalize()
    print(f"{dim_label}: QWK={qwk:.4f}, spearman={spearman_rho:.4f} (p={spearman_p:.4e}), exact={exact_agreement_pct:.1%}, within-1={within_one_pct:.1%}")

macro_avg_qwk= np.mean([v['qwk'] for v in per_dimension_agreement_results.values()])
print(f"\nThe macro-average QWK is {macro_avg_qwk:.4f}")
print("from the classic landis & koch benchmarks: <0.20 is slight, 0.21-0.40 is fair, 0.41-0.60 is moderate, 0.61-0.80 substantial & >0.80 almost perfect")

## Confusion matrices

I made these to show the joint distribution of original vs re-annotated ratings per dimension, with the idea being they reveal if disagreements are systematic (e.g. consistent upward drift on the second pass) or random noise around borderline cases.

In [ ]:
for dim in agreement_config['score_cols']:
    original_ratings= merged_annotations_df[f'{dim}_original'].astype(int).values
    my_reannot_ratings= merged_annotations_df[f'{dim}_reannot'].astype(int).values

    confusion_mat_df= pd.DataFrame(
        confusion_matrix(original_ratings, my_reannot_ratings, labels=agreement_config['likert_labels']),
        index=[f'orig_{i}' for i in agreement_config['likert_labels']], columns=[f're_{i}' for i in agreement_config['likert_labels']])
    
    print(f"\n{dim.replace('_score', '').capitalize()} (rows=original, cols=re-annotation)")
    print(confusion_mat_df.to_string())

## High-discrepancy sentences

These are sentences where the two passes disagreed by 2 or more scale points on any dimension, are they are worth looking at. They could be useful for identifying where the codebook is ambiguous.

In [ ]:
merged_annotations_df['max_discrepancy_across_dimensions']= 0
for dim in agreement_config['score_cols']:
    diff_in_scores= np.abs(merged_annotations_df[f'{dim}_original'] - merged_annotations_df[f'{dim}_reannot'])
    merged_annotations_df['max_discrepancy_across_dimensions']= np.maximum(merged_annotations_df['max_discrepancy_across_dimensions'], diff_in_scores)

high_discrepancy_sentences= merged_annotations_df[merged_annotations_df['max_discrepancy_across_dimensions'] >= agreement_config['high_discrepancy_threshold']].copy()
print(f"sentences with discrepancy >= {agreement_config['high_discrepancy_threshold']} on any dimension are these ones: {len(high_discrepancy_sentences)}/{len(merged_annotations_df)}")

# now I print the first 10 high-discrepancy sentences with their scores side-by-side
for i, (_, row) in enumerate(high_discrepancy_sentences.head(10).iterrows(), 1):
    print(f"{i}. {row['sentence_text'][:100]}")
    for dim in agreement_config['score_cols']:
        dim_score= dim.replace('_score', '')
        orig= int(row[f'{dim}_original'])
        my_reannot= int(row[f'{dim}_reannot'])
        print(f"{dim_score}: {orig} -> {my_reannot}")
    print()